# Exploring ListenBrainz Listen Data

General overview of the Spark dump data, followed by RecSys-readiness analysis.

In [ ]:
import pyarrow.compute as pc
import matplotlib.pyplot as plt
from music_recommendation.dataloader import load_listens

plt.style.use("ggplot")

listens = load_listens("../data/listenbrainz")
print(f"Loaded {listens.num_rows:,} listens")

## 1. General Overview

In [ ]:
n = listens.num_rows
ts = listens.column("listened_at")
mbid = listens.column("recording_mbid")

mbid_non_null = n - pc.sum(pc.is_null(mbid)).as_py()

print(f"Total listens:        {n:,}")
print(f"Date range:           {pc.min(ts).as_py()} to {pc.max(ts).as_py()}")
print(f"Unique users:         {pc.count_distinct(listens.column('user_id')).as_py():,}")
print(f"Unique recording_msid: {pc.count_distinct(listens.column('recording_msid')).as_py():,}")
print(f"Unique recording_mbid: {pc.count_distinct(mbid).as_py():,} (non-null)")
print(f"recording_mbid coverage: {mbid_non_null:,} / {n:,} ({mbid_non_null*100/n:.1f}%)")

### Listens per hour

When do people listen to music? This histogram buckets every listen by hour of day (UTC). Peaks suggest global prime-time listening windows; the trough typically falls during early-morning hours in Europe/Americas.

In [ ]:
hours = pc.hour(ts).to_pylist()

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(hours, bins=range(25), edgecolor="white", alpha=0.8)
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("Listen count")
ax.set_title("Listen distribution by hour of day")
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

## 2. User Activity Distribution

How active are users? The left plot shows the raw distribution of listens per user — most users have relatively few listens while a handful are extremely active. The right plot uses a log scale on the y-axis to reveal the shape of the long tail.

In [ ]:
user_counts = listens.group_by("user_id").aggregate([("recording_msid", "count")])
counts = sorted(user_counts.column("recording_msid_count").to_pylist(), reverse=True)

print(f"Users: {len(counts):,}")
print(f"  Min listens:    {counts[-1]}")
print(f"  Median listens: {counts[len(counts)//2]}")
print(f"  Mean listens:   {sum(counts)/len(counts):.0f}")
print(f"  Max listens:    {counts[0]}")
print(f"  Users with 1 listen: {sum(1 for c in counts if c == 1):,}")
print(f"  Users with >=10 listens: {sum(1 for c in counts if c >= 10):,}")
print(f"  Users with >=50 listens: {sum(1 for c in counts if c >= 50):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(counts, bins=100, edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Listens per user")
axes[0].set_ylabel("Number of users")
axes[0].set_title("User activity distribution")

axes[1].hist(counts, bins=100, edgecolor="white", alpha=0.8, log=True)
axes[1].set_xlabel("Listens per user")
axes[1].set_ylabel("Number of users (log)")
axes[1].set_title("User activity distribution (log scale)")

plt.tight_layout()
plt.show()

## 3. Item Popularity Distribution

How are listens distributed across tracks? This log-log rank plot shows items sorted by popularity. A straight line on a log-log scale indicates a power-law distribution — a small number of hits dominate while the vast majority of tracks are rarely played.

In [ ]:
# Use recording_mbid where available
mbid_listens = listens.filter(pc.is_valid(listens.column("recording_mbid")))
item_counts = mbid_listens.group_by("recording_mbid").aggregate([("user_id", "count")])
item_pops = sorted(item_counts.column("user_id_count").to_pylist(), reverse=True)

print(f"Unique items (mbid): {len(item_pops):,}")
print(f"  Min listens:    {item_pops[-1]}")
print(f"  Median listens: {item_pops[len(item_pops)//2]}")
print(f"  Mean listens:   {sum(item_pops)/len(item_pops):.1f}")
print(f"  Max listens:    {item_pops[0]}")
print(f"  Items with 1 listen:   {sum(1 for c in item_pops if c == 1):,}")
print(f"  Items with >=5 listens: {sum(1 for c in item_pops if c >= 5):,}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(item_pops) + 1), item_pops)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Item rank")
ax.set_ylabel("Listen count")
ax.set_title("Item popularity (log-log) — long tail distribution")
plt.tight_layout()
plt.show()

## 4. Interaction Matrix Sparsity

The interaction matrix has one row per user and one column per item. Each cell is filled if that user has listened to that item at least once. This chart compares the number of filled cells (actual interactions) to the number of empty cells — the overwhelming majority are empty, which is typical for recommendation datasets.

In [ ]:
n_users = pc.count_distinct(mbid_listens.column("user_id")).as_py()
n_items = len(item_pops)
# Count unique (user, item) pairs
pairs = mbid_listens.select(["user_id", "recording_mbid"]).group_by(["user_id", "recording_mbid"]).aggregate([])
n_interactions = pairs.num_rows

sparsity = 1 - n_interactions / (n_users * n_items)

print(f"Users (with mbid listens): {n_users:,}")
print(f"Items (unique mbid):       {n_items:,}")
print(f"Unique (user, item) pairs: {n_interactions:,}")
print(f"Possible pairs:            {n_users * n_items:,}")
print(f"Sparsity:                  {sparsity*100:.4f}%")
print(f"Density:                   {(1-sparsity)*100:.6f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Interactions", "Empty"],
    [n_interactions, n_users * n_items - n_interactions],
    color=["#4C72B0", "#DD8452"],
    edgecolor="white",
)
ax.set_yscale("log")
ax.set_ylabel("Count (log)")
ax.set_title(f"Interaction matrix: {(1-sparsity)*100:.4f}% density")
for bar, val in zip(bars, [n_interactions, n_users * n_items - n_interactions]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:,.0f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Cold-Start Analysis

Cold-start is a core challenge for recommender systems: users or items with very few interactions are hard to make good predictions for. The left plot shows how many unique items each user has listened to — users below the red threshold line may not have enough signal. The right plot shows how many unique users each item has been listened to by — most items are listened to by very few users.

In [ ]:
# Users with very few unique items
user_unique_items = pairs.group_by("user_id").aggregate([("recording_mbid", "count")])
unique_item_counts = user_unique_items.column("recording_mbid_count").to_pylist()

thresholds = [1, 2, 5, 10, 20]
print("Cold-start users (few unique items listened):")
for t in thresholds:
    count = sum(1 for c in unique_item_counts if c < t)
    print(f"  <{t:>2} unique items: {count:>7,} users ({count*100/len(unique_item_counts):.1f}%)")

print()

# Items listened by very few users
item_unique_users = pairs.group_by("recording_mbid").aggregate([("user_id", "count")])
unique_user_counts = item_unique_users.column("user_id_count").to_pylist()

print("Cold-start items (few unique users):")
for t in thresholds:
    count = sum(1 for c in unique_user_counts if c < t)
    print(f"  <{t:>2} unique users: {count:>7,} items ({count*100/len(unique_user_counts):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(unique_item_counts, bins=100, edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Unique items per user")
axes[0].set_ylabel("Number of users")
axes[0].set_title("User cold-start: unique items per user")
axes[0].axvline(x=20, color="red", linestyle="--", label="threshold=20")
axes[0].legend()

axes[1].hist(unique_user_counts, bins=100, edgecolor="white", alpha=0.8, log=True)
axes[1].set_xlabel("Unique users per item")
axes[1].set_ylabel("Number of items (log)")
axes[1].set_title("Item cold-start: unique users per item")
axes[1].axvline(x=5, color="red", linestyle="--", label="threshold=5")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Repeat Listening Behavior

Do users re-listen to the same tracks? The left plot shows the distribution of play counts per (user, item) pair up to 50 plays. The right plot uses log scale to show the full range. A high share of repeat listens suggests that play count can serve as an implicit signal of preference strength.

In [ ]:
# How often do users re-listen to the same item?
user_item_counts = mbid_listens.group_by(["user_id", "recording_mbid"]).aggregate([("listened_at", "count")])
replay_counts = user_item_counts.column("listened_at_count").to_pylist()

single = sum(1 for c in replay_counts if c == 1)
total_pairs = len(replay_counts)

print(f"Unique (user, item) pairs: {total_pairs:,}")
print(f"  Listened once:     {single:,} ({single*100/total_pairs:.1f}%)")
print(f"  Listened 2+ times: {total_pairs - single:,} ({(total_pairs-single)*100/total_pairs:.1f}%)")
print(f"  Mean plays per pair: {sum(replay_counts)/total_pairs:.2f}")
print(f"  Max plays per pair:  {max(replay_counts)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(replay_counts, bins=range(1, 52), edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Times played")
axes[0].set_ylabel("Number of (user, item) pairs")
axes[0].set_title("Repeat listening distribution (1–50 plays)")
axes[0].set_xlim(1, 50)

capped = [min(c, 200) for c in replay_counts]
axes[1].hist(capped, bins=100, edgecolor="white", alpha=0.8, log=True)
axes[1].set_xlabel("Times played (capped at 200)")
axes[1].set_ylabel("Number of pairs (log)")
axes[1].set_title("Repeat listening distribution (log scale)")

plt.tight_layout()
plt.show()